# AIAA 3102 Assignment 1 — CIFAR-10 classification

**Student:** LIU, Yixuan — **Student ID:** 50027321

## Final method and rationale

We learn local patch dictionaries exclusively from the provided training images.
Local contrast normalization and regularized ZCA whitening precede MiniBatchKMeans;
triangle coding and quadrant pooling preserve local image structure. HOG and
spatial color statistics add complementary information. Scikit-learn MLP classifiers
learn from clean images and seeded, label-preserving corruption augmentations.
The frozen configuration below specifies all model members, training epochs,
ensemble weights, and whether horizontal-flip prediction averaging is enabled.

The ensemble has seven members. Four differ in patch size, dictionary size,
smoothing, and augmentation strength. Three see a fixed preprocessed image
before all features: a 3x3 median filter, grayscale, or 2x2 low resolution.
This diversity makes members fail on different corruptions. On the development
subset, the seven-member ensemble reached 66.77% versus 65.80% for the previous
four-member recipe (McNemar p = 0.014), while extra test-time views and a
broader augmentation did not help.

Development uses a fixed 45,000/5,000 split of the provided training set and a
3,500/1,500 development/audit split of the official validation set. The audit labels
are not used to choose the model. All final components are retrained from scratch
on the 50,000 training images for each required seed; official validation images
are used only for reporting. No external data, pretrained weights, or custom
classification algorithm is used. Feature learning is preprocessing; all supervised
learners are `sklearn.neural_network.MLPClassifier` instances.

To avoid CPU slowdowns from float32 subnormals, values below `1e-20` in weights
and optimizer state are rounded to zero between epochs. Development prediction
probabilities are unchanged by this cleanup. Training-only optimizer state is
omitted from exported files; prediction equality is checked before each export.

The patch-feature design is inspired by Coates, Lee and Ng (AISTATS 2011):
https://proceedings.mlr.press/v15/coates11a.html . Paper results are not our results.

## How to run

Use Python 3.14.6, NumPy 2.4.6, SciPy 1.18.0, scikit-learn 1.9.0 and joblib 1.5.3,
with a Jupyter kernel for that environment. Set `DATA_ROOT` below if needed. The
default expects `student_data/CIFAR10_Data` and
`student_data/validation_set_to_students` next to this notebook (or one folder up).
Training uses CPU BLAS, not CUDA. Approximately 16 GB free RAM is recommended. Training one seed at a time takes
several hours on a 6-thread CPU; actual time depends on the machine and is printed below.
Execution defaults to one seed at a time. On a machine with at least 32 GB free
RAM, set the environment variable `ASSIGNMENT1_N_JOBS=3` before launching Jupyter
to train the three independent seeds in parallel. This does not change the recipe.

**Run All rebuilds all three models from raw course data.** No experiment cache,
existing model file, local Python source file, internet connection, or user input
is required. The outputs are the three required `.joblib` files in the working
directory. Model prediction includes preprocessing and needs no training files.
Custom preprocessing code is embedded using joblib's cloudpickle wrapper; the
exported files are checked in a new interpreter without importing notebook code.


In [1]:
import os
os.environ['LOKY_MAX_CPU_COUNT'] = '6'
os.environ['OMP_NUM_THREADS'] = '6'
os.environ['OPENBLAS_NUM_THREADS'] = '6'
import sys, platform, json, time, hashlib, tempfile, subprocess
from pathlib import Path
import numpy as np
import scipy, sklearn, joblib
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
versions = {'python': platform.python_version(), 'numpy': np.__version__,
            'scipy': scipy.__version__, 'scikit-learn': sklearn.__version__, 'joblib': joblib.__version__}
expected = {'python':'3.14.6','numpy':'2.4.6','scipy':'1.18.0','scikit-learn':'1.9.0','joblib':'1.5.3'}
print(json.dumps(versions, indent=2))
assert versions == expected, 'Use the course evaluation environment before exporting final models.'
DATA_ROOT = Path('student_data')  # EDIT HERE if your extracted data is elsewhere.
if not (DATA_ROOT / 'CIFAR10_Data/x_train.npy').exists():
    for candidate in [Path('../student_data'), Path('.')]:
        if (candidate / 'CIFAR10_Data/x_train.npy').exists():
            DATA_ROOT = candidate
            break
assert (DATA_ROOT / 'CIFAR10_Data/x_train.npy').exists(), 'Extract student_data.zip and configure DATA_ROOT.'
X_train = np.load(DATA_ROOT / 'CIFAR10_Data/x_train.npy')
y_train = np.load(DATA_ROOT / 'CIFAR10_Data/y_train.npy').ravel()
X_validation = np.load(DATA_ROOT / 'validation_set_to_students/images.npy')
y_validation = np.load(DATA_ROOT / 'validation_set_to_students/labels.npy').ravel()
assert X_train.shape == (50000,32,32,3) and y_train.shape == (50000,)
assert X_validation.shape == (5000,32,32,3) and y_validation.shape == (5000,)
assert np.array_equal(np.unique(y_train), np.arange(10))
assert X_train.dtype == np.uint8 and X_validation.dtype == np.uint8
print('Training:', X_train.shape, 'classes:', np.bincount(y_train))
print('Validation:', X_validation.shape, 'classes:', np.bincount(y_validation))
dev_indices, audit_indices = train_test_split(np.arange(5000), test_size=1500,
    stratify=y_validation, random_state=1729)


{
  "python": "3.14.6",
  "numpy": "2.4.6",
  "scipy": "1.18.0",
  "scikit-learn": "1.9.0",
  "joblib": "1.5.3"
}
Training: (50000, 32, 32, 3) classes: [5000 5000 5000 5000 5000 5000 5000 5000 5000 5000]
Validation: (5000, 32, 32, 3) classes: [500 500 500 500 500 500 500 500 500 500]


## Self-contained preprocessing and export implementation

In [2]:
"""Self-contained CIFAR feature learning; no external data or model weights.

This source is embedded verbatim in the final notebook. Inference depends only on
NumPy, SciPy, scikit-learn and joblib. Training data and caches are never needed
by the exported model.
"""
import time
import numpy as np
from scipy.ndimage import gaussian_filter
from sklearn.cluster import MiniBatchKMeans
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.base import BaseEstimator, ClassifierMixin
from threadpoolctl import threadpool_limits


def preprocess_images(images, mode):
    """Fixed per-image view seen by a member before all features (float32 output).

    'median3' is a 3x3 spatial median, 'gray' replicates luminance to three
    channels, and 'lowres' averages 2x2 blocks then repeats them. Each view makes
    one ensemble member insensitive to a different corruption family.
    """
    import numpy as np
    from scipy.ndimage import median_filter
    x = np.asarray(images, dtype=np.float32)
    if mode == 'median3':
        return median_filter(x, size=(1, 3, 3, 1))
    if mode == 'gray':
        return np.repeat((x @ np.float32([.299, .587, .114]))[..., None], 3, axis=3)
    if mode == 'lowres':
        x = x.reshape(len(x), 16, 2, 16, 2, 3).mean(axis=(2, 4))
        return np.repeat(np.repeat(x, 2, axis=1), 2, axis=2)
    raise ValueError(f'Unknown preprocessing: {mode}')


def learn_dictionary(images, seed, n_clusters=512, patch_size=6, n_patches=60000, smooth_sigma=0,
                     preprocess=None):
    """Fit local contrast normalization, regularized ZCA, and a patch dictionary."""
    rng = np.random.default_rng(seed)
    if preprocess:
        images = preprocess_images(images, preprocess)
    if smooth_sigma:
        images = gaussian_filter(images.astype(np.float32), sigma=(0, smooth_sigma, smooth_sigma, 0))
    count = len(images)
    indices = rng.integers(count, size=n_patches)
    rows = rng.integers(33 - patch_size, size=n_patches)
    cols = rng.integers(33 - patch_size, size=n_patches)
    patches = np.empty((n_patches, patch_size * patch_size * 3), dtype=np.float32)
    for i, (index, row, col) in enumerate(zip(indices, rows, cols)):
        patch = images[index, row:row + patch_size, col:col + patch_size]
        if rng.random() < .5:
            patch = patch[:, ::-1]
        patches[i] = patch.reshape(-1)
    patches -= patches.mean(axis=1, keepdims=True)
    patches /= np.sqrt(patches.var(axis=1, keepdims=True) + 10.0)
    mean = patches.mean(axis=0)
    patches -= mean
    cov = (patches.T @ patches) / len(patches)
    values, vectors = np.linalg.eigh(cov)
    whitening = ((vectors * (1.0 / np.sqrt(np.maximum(values, 0) + .1))) @ vectors.T).astype(np.float32)
    patches = np.ascontiguousarray(patches @ whitening)
    km = MiniBatchKMeans(n_clusters=n_clusters, batch_size=2048,
                        n_init=3, max_iter=80, max_no_improvement=15,
                        random_state=seed, reassignment_ratio=.01)
    started = time.perf_counter()
    with threadpool_limits(limits=6):
        km.fit(patches)
    print(f'Dictionary K={n_clusters}, p={patch_size}: {time.perf_counter()-started:.1f}s', flush=True)
    return dict(centers=km.cluster_centers_.astype(np.float32), mean=mean,
                whitening=whitening, patch_size=patch_size, smooth_sigma=smooth_sigma,
                preprocess=preprocess)


def image_features(images, banks=(), include_hog=True, stride=2, batch_size=48):
    """Deterministic per-image transform, with bounded temporary allocations.

    All imports and calculations needed by prediction are inside this function.
    The HOG implementation uses unsigned, linearly interpolated orientation bins
    and L2-Hys block normalization. Learned features use triangle coding and
    quadrant average pooling, followed by a square-root transform.
    """
    import numpy as np
    from scipy.ndimage import gaussian_filter
    from threadpoolctl import threadpool_limits
    x = np.asarray(images)
    if x.ndim != 4 or x.shape[1:] != (32, 32, 3):
        raise ValueError('Expected images with shape (N, 32, 32, 3).')
    if len(x) == 0:
        dimension = sum(4 * len(bank['centers']) for bank in banks) + 96
        if include_hog:
            dimension += 1764
        return np.empty((0, dimension), dtype=np.float32)
    result = []
    with threadpool_limits(limits=6):
        for start in range(0, len(x), batch_size):
            batch = x[start:start + batch_size].astype(np.float32)
            mode = banks[0].get('preprocess') if banks else None
            if mode:
                batch = preprocess_images(batch, mode)
            sigma = banks[0].get('smooth_sigma', 0) if banks else 0
            if sigma:
                batch = gaussian_filter(batch, sigma=(0, sigma, sigma, 0))
            n = len(batch)
            parts = []
            for bank in banks:
                p = bank['patch_size']
                view = np.lib.stride_tricks.sliding_window_view(batch, (p, p), axis=(1, 2))
                view = view[:, ::stride, ::stride].transpose(0, 1, 2, 4, 5, 3)
                h, w = view.shape[1:3]
                patches = np.ascontiguousarray(view).reshape(-1, p * p * 3)
                patches -= patches.mean(axis=1, keepdims=True)
                patches /= np.sqrt(patches.var(axis=1, keepdims=True) + 10.0)
                patches = (patches - bank['mean']) @ bank['whitening']
                centers = bank['centers']
                distances = patches @ centers.T
                distances *= -2
                distances += np.sum(patches * patches, axis=1, keepdims=True)
                distances += np.sum(centers * centers, axis=1)
                np.maximum(distances, 0, out=distances)
                np.sqrt(distances, out=distances)
                activation = np.maximum(distances.mean(axis=1, keepdims=True) - distances, 0)
                activation = activation.reshape(n, h, w, -1)
                quadrants = [activation[:, a:b, c:d].mean(axis=(1, 2))
                             for a, b in ((0, h//2), (h//2, h))
                             for c, d in ((0, w//2), (w//2, w))]
                parts.append(np.sqrt(np.concatenate(quadrants, axis=1)))
            # Spatial color means and standard deviations, 4x4 regions.
            color = batch.reshape(n, 4, 8, 4, 8, 3)
            parts.extend([color.mean(axis=(2, 4)).reshape(n, -1) / 255,
                          color.std(axis=(2, 4)).reshape(n, -1) / 255])
            if include_hog:
                # Choose the strongest color-channel gradient at each pixel.
                dx = np.zeros_like(batch)
                dy = np.zeros_like(batch)
                dx[:, :, 1:-1] = batch[:, :, 2:] - batch[:, :, :-2]
                dy[:, 1:-1] = batch[:, 2:] - batch[:, :-2]
                mag2 = dx*dx + dy*dy
                ch = mag2.argmax(axis=3)[..., None]
                gx = np.take_along_axis(dx, ch, axis=3)[..., 0]
                gy = np.take_along_axis(dy, ch, axis=3)[..., 0]
                mag = np.sqrt(np.take_along_axis(mag2, ch, axis=3)[..., 0])
                angle = (np.arctan2(gy, gx) % np.pi) * (9 / np.pi)
                lo = np.floor(angle).astype(np.int32)
                frac = angle - lo
                hist = np.empty((n, 8, 8, 9), np.float32)
                for k in range(9):
                    votes = mag * ((lo % 9 == k) * (1-frac) + ((lo+1) % 9 == k) * frac)
                    hist[..., k] = votes.reshape(n, 8, 4, 8, 4).sum(axis=(2, 4))
                blocks = np.lib.stride_tricks.sliding_window_view(hist, (2, 2), axis=(1, 2))
                blocks = blocks.transpose(0, 1, 2, 4, 5, 3).reshape(n, 7, 7, 36)
                blocks = blocks / np.sqrt(np.sum(blocks*blocks, axis=3, keepdims=True) + 1e-6)
                blocks = np.minimum(blocks, .2)
                blocks /= np.sqrt(np.sum(blocks*blocks, axis=3, keepdims=True) + 1e-6)
                parts.append(blocks.reshape(n, -1))
            result.append(np.concatenate(parts, axis=1).astype(np.float32))
    return np.concatenate(result)


def augment_images(images, seed, mode='mixed'):
    """Seeded, label-preserving augmentation of training images only."""
    rng = np.random.default_rng(seed)
    if mode == 'flip':
        return np.ascontiguousarray(images[:, :, ::-1])
    x = images.astype(np.float32).copy()
    flips = rng.random(len(x)) < .5
    x[flips] = x[flips, :, ::-1]
    for i in range(len(x)):
        dy, dx = rng.integers(-2, 3, size=2)
        padded = np.pad(x[i], ((2, 2), (2, 2), (0, 0)), mode='reflect')
        x[i] = padded[2+dy:34+dy, 2+dx:34+dx]
    if mode in ('robust', 'robust2'):
        strong = mode == 'robust2'
        for i in range(len(x)):
            kind = rng.integers(0, 7)
            if kind == 1:
                x[i] += rng.normal(0, rng.uniform(4, 40 if strong else 24), size=x[i].shape)
            elif kind == 2:
                x[i] = gaussian_filter(x[i], sigma=(rng.uniform(.4, 1.8 if strong else 1.15),) * 2 + (0,))
            elif kind == 3:
                x[i] *= rng.uniform(.45 if strong else .65, 1.55 if strong else 1.35, size=(1, 1, 3))
                x[i] += rng.uniform(-20, 20)
            elif kind == 4:
                mask = rng.random((32, 32)) < rng.uniform(.01, .10 if strong else .05)
                x[i][mask] = rng.choice([0, 255], size=(mask.sum(), 1))
            elif kind == 5:
                reduced = x[i].reshape(16, 2, 16, 2, 3).mean(axis=(1, 3))
                x[i] = np.repeat(np.repeat(reduced, 2, axis=0), 2, axis=1)
            elif kind == 6:
                size = int(rng.integers(4, 9))
                r, c = rng.integers(0, 33-size, size=2)
                x[i, r:r+size, c:c+size] = rng.integers(0, 256, size=(1, 1, 3))
        return np.clip(x, 0, 255).astype(np.uint8)
    # Keep several types separate; avoid always compounding all corruptions.
    kind = rng.integers(0, 4, size=len(x))
    for k in range(4):
        idx = np.flatnonzero(kind == k)
        if k == 1:
            x[idx] += rng.normal(0, 6, size=x[idx].shape).astype(np.float32)
        elif k == 2:
            x[idx] = gaussian_filter(x[idx], sigma=(0, .55, .55, 0))
        elif k == 3:
            x[idx] = x[idx] * rng.uniform(.8, 1.2, size=(len(idx), 1, 1, 1))
            x[idx] += rng.uniform(-12, 12, size=(len(idx), 1, 1, 1))
    return np.clip(x, 0, 255).astype(np.uint8)


def make_pipeline(banks, classifier, scaler=None, include_hog=True):
    transform = FunctionTransformer(image_features,
        kw_args=dict(banks=banks, include_hog=include_hog), validate=False)
    if scaler is None:
        scaler = StandardScaler()
    return Pipeline([('features', transform), ('scale', scaler), ('classifier', classifier)])


def strip_training_state(classifier):
    """Drop Adam moments and unused early-stopping backups, retaining all weights.

    These attributes are only used for continuing training, never prediction.
    The submission is an inference artifact; Run All always trains from scratch.
    Prediction equality before/after stripping is tested before every export.
    """
    for attribute in ('_optimizer', '_best_coefs', '_best_intercepts'):
        if hasattr(classifier, attribute):
            delattr(classifier, attribute)
    return classifier


def flush_tiny_training_values(classifier, threshold=1e-20):
    """Round numerically negligible float32 values to zero between epochs.

    Dead ReLU units plus Adam/regularization can generate CPU-slow subnormals.
    The threshold is far below float32 prediction precision. Apply identically
    for every seed, and test prediction equality for the developed candidates.
    """
    arrays = list(classifier.coefs_) + list(classifier.intercepts_)
    optimizer = getattr(classifier, '_optimizer', None)
    if optimizer is not None:
        arrays += list(getattr(optimizer, 'ms', [])) + list(getattr(optimizer, 'vs', []))
    for array in arrays:
        array[np.abs(array) < threshold] = 0
    return classifier


def save_portable(model, path):
    """Embed custom feature code, without requiring assignment_model.py at load."""
    import sys
    import joblib
    from joblib.externals import cloudpickle
    from joblib import wrap_non_picklable_objects
    module = sys.modules.get(image_features.__module__)
    registered = module is not None and module.__name__ != '__main__'
    if registered:
        cloudpickle.register_pickle_by_value(module)
    try:
        joblib.dump(wrap_non_picklable_objects(model), path, compress=3, protocol=5)
    finally:
        if registered:
            cloudpickle.unregister_pickle_by_value(module)


class ImageClassifier(ClassifierMixin, BaseEstimator):
    """Average fitted sklearn probability pipelines and optional horizontal views.

    Members are independently fitted with the frozen training recipe before this
    inference-only container is constructed. No model is fitted at prediction.
    """
    def __init__(self, members, weights=None, horizontal_tta=False):
        self.members = members
        self.weights = weights
        self.horizontal_tta = horizontal_tta
        self.classes_ = np.asarray(members[0].classes_)
        for member in members:
            if not np.array_equal(member.classes_, self.classes_):
                raise ValueError('All members must have identical class ordering.')

    def predict_proba(self, images):
        weights = np.ones(len(self.members)) if self.weights is None else np.asarray(self.weights)
        if len(weights) != len(self.members) or np.any(weights < 0) or weights.sum() <= 0:
            raise ValueError('Invalid ensemble weights.')
        result = None
        for weight, model in zip(weights / weights.sum(), self.members):
            probability = model.predict_proba(images)
            if self.horizontal_tta:
                probability = (probability + model.predict_proba(np.asarray(images)[:, :, ::-1])) * .5
            result = weight * probability if result is None else result + weight * probability
        return result

    def predict(self, images):
        return self.classes_[self.predict_proba(images).argmax(axis=1)]


## Frozen training recipe
All three seeds use the same settings. Validation scores do not control final training.

In [3]:
CONFIGURATION = {'selection_metric': '3500-image official development accuracy', 'selection_accuracy': 0.6677, 'audit_used_for_selection': False, 'training_threads': 1, 'tiny_value_threshold': 1e-20, 'horizontal_tta': True, 'weights': [1, 1, 1, 1, 1, 1, 1], 'members': [{'name': 'raw', 'clusters': 512, 'patch_size': 6, 'n_patches': 60000, 'smooth_sigma': 0, 'augmentation_mode': 'robust', 'augmentation_ratio': 2.0, 'hidden_layers': [384, 128], 'alpha': 5.0, 'learning_rate': 0.0005, 'epochs': 5}, {'name': 'strong', 'clusters': 512, 'patch_size': 6, 'n_patches': 60000, 'smooth_sigma': 0, 'augmentation_mode': 'robust2', 'augmentation_ratio': 2.0, 'hidden_layers': [384, 128], 'alpha': 5.0, 'learning_rate': 0.0005, 'epochs': 15}, {'name': 'largepatch', 'clusters': 512, 'patch_size': 8, 'n_patches': 60000, 'smooth_sigma': 0, 'augmentation_mode': 'robust', 'augmentation_ratio': 2.0, 'hidden_layers': [384, 128], 'alpha': 5.0, 'learning_rate': 0.0005, 'epochs': 15}, {'name': 'smooth', 'clusters': 1024, 'patch_size': 6, 'n_patches': 60000, 'smooth_sigma': 0.6, 'augmentation_mode': 'robust', 'augmentation_ratio': 1.0, 'hidden_layers': [384, 128], 'alpha': 5.0, 'learning_rate': 0.0005, 'epochs': 23}, {'name': 'median3', 'clusters': 512, 'patch_size': 6, 'n_patches': 60000, 'smooth_sigma': 0, 'preprocess': 'median3', 'augmentation_mode': 'robust', 'augmentation_ratio': 2.0, 'hidden_layers': [384, 128], 'alpha': 5.0, 'learning_rate': 0.0005, 'epochs': 15}, {'name': 'gray', 'clusters': 512, 'patch_size': 6, 'n_patches': 60000, 'smooth_sigma': 0, 'preprocess': 'gray', 'augmentation_mode': 'robust', 'augmentation_ratio': 2.0, 'hidden_layers': [384, 128], 'alpha': 5.0, 'learning_rate': 0.0005, 'epochs': 14}, {'name': 'lowres', 'clusters': 512, 'patch_size': 6, 'n_patches': 60000, 'smooth_sigma': 0, 'preprocess': 'lowres', 'augmentation_mode': 'robust', 'augmentation_ratio': 2.0, 'hidden_layers': [384, 128], 'alpha': 5.0, 'learning_rate': 0.0005, 'epochs': 7}]}
print(json.dumps(CONFIGURATION, indent=2))
results = []


{
  "selection_metric": "3500-image official development accuracy",
  "selection_accuracy": 0.6677,
  "audit_used_for_selection": false,
  "training_threads": 1,
  "tiny_value_threshold": 1e-20,
  "horizontal_tta": true,
  "weights": [
    1,
    1,
    1,
    1,
    1,
    1,
    1
  ],
  "members": [
    {
      "name": "raw",
      "clusters": 512,
      "patch_size": 6,
      "n_patches": 60000,
      "smooth_sigma": 0,
      "augmentation_mode": "robust",
      "augmentation_ratio": 2.0,
      "hidden_layers": [
        384,
        128
      ],
      "alpha": 5.0,
      "learning_rate": 0.0005,
      "epochs": 5
    },
    {
      "name": "strong",
      "clusters": 512,
      "patch_size": 6,
      "n_patches": 60000,
      "smooth_sigma": 0,
      "augmentation_mode": "robust2",
      "augmentation_ratio": 2.0,
      "hidden_layers": [
        384,
        128
      ],
      "alpha": 5.0,
      "learning_rate": 0.0005,
      "epochs": 15
    },
    {
      "name": "largepatch",

In [4]:
"""Final training recipe; this entire file is embedded in train_models.ipynb."""
import gc
import time
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits


def fit_final_seed(images, labels, seed, configuration):
    """Retrain every learned component from scratch, with no development caches."""
    members = []
    timings = []
    # Within one seed only: identical dictionary recipes share exact features.
    # This cache is rebuilt from raw training data on every Run All and seed.
    dictionary_cache = {}
    feature_cache = {}
    key_of = lambda s: (s['clusters'], s.get('patch_size', 6), s.get('n_patches', 60000),
                        s['smooth_sigma'], s.get('preprocess'))
    keys = [key_of(specification) for specification in configuration['members']]
    for index, specification in enumerate(configuration['members']):
        started = time.perf_counter()
        key = keys[index]
        if key not in dictionary_cache:
            print(f'seed={seed} learning dictionary {key}', flush=True)
            dictionary_cache[key] = learn_dictionary(images, seed,
                n_clusters=key[0], patch_size=key[1], n_patches=key[2], smooth_sigma=key[3],
                preprocess=key[4])
            feature_cache[key] = image_features(images, banks=[dictionary_cache[key]])
            print(f'seed={seed} original features ready: {feature_cache[key].shape}', flush=True)
        bank = dictionary_cache[key]
        original_features = feature_cache[key]
        if key not in keys[index + 1:]:
            del feature_cache[key]  # Bound peak memory; no later member needs it.
        ratio = specification['augmentation_ratio']
        rng = np.random.default_rng(seed)
        chosen = rng.choice(len(images), size=int(len(images) * ratio), replace=ratio > 1)
        augmented = augment_images(images[chosen], seed, mode=specification.get('augmentation_mode', 'robust'))
        augmented_features = image_features(augmented, banks=[bank])
        print(f'seed={seed} augmented features ready: {augmented_features.shape}', flush=True)
        del augmented
        features = np.concatenate([original_features, augmented_features])
        targets = np.concatenate([labels, labels[chosen]])
        del original_features, augmented_features
        scaler = StandardScaler(copy=False)
        features = scaler.fit_transform(features)
        classifier = MLPClassifier(hidden_layer_sizes=tuple(specification['hidden_layers']),
            alpha=specification['alpha'], learning_rate_init=specification['learning_rate'],
            batch_size=512, max_iter=1, random_state=seed, shuffle=True)
        for epoch in range(specification['epochs']):
            with threadpool_limits(limits=configuration.get('training_threads', 1)):
                classifier.partial_fit(features, targets, classes=np.arange(10))
            flush_tiny_training_values(classifier, configuration.get('tiny_value_threshold', 1e-20))
            print(f"seed={seed} K={specification['clusters']} epoch={epoch+1}/{specification['epochs']} loss={classifier.loss_:.4f}", flush=True)
        before = classifier.predict_proba(features[:37])
        strip_training_state(classifier)
        np.testing.assert_array_equal(before, classifier.predict_proba(features[:37]))
        del features, targets
        members.append(make_pipeline([bank], classifier, scaler=scaler))
        elapsed = time.perf_counter() - started
        timings.append(round(elapsed, 2))
        print(f'Member completed in {elapsed:.1f}s', flush=True)
        gc.collect()
    return ImageClassifier(members, weights=configuration['weights'],
                           horizontal_tta=configuration['horizontal_tta']), timings


## Train the three independent seeds from scratch

In [5]:
from joblib import Parallel, delayed
SEEDS = [42, 2026, 3102]
N_JOBS = int(os.environ.get('ASSIGNMENT1_N_JOBS', '1'))
assert N_JOBS in [1, 2, 3]
training_started = time.perf_counter()
print('Independent seed workers:', N_JOBS, flush=True)
with tempfile.TemporaryDirectory(dir=Path.cwd(), prefix='training_temp_') as cache_directory:
    fitted_models = Parallel(n_jobs=N_JOBS, backend='loky', temp_folder=cache_directory)(
        delayed(fit_final_seed)(X_train, y_train, seed, CONFIGURATION) for seed in SEEDS)
training_wall_seconds = time.perf_counter() - training_started
print('All three seeds trained from scratch in', round(training_wall_seconds, 2), 'seconds')


Independent seed workers: 3


All three seeds trained from scratch in 2953.32 seconds


## Export and check seed 42

In [6]:
seed = 42
started = time.perf_counter()
model, member_seconds = fitted_models.pop(0)
model_path = Path(f'model_seed{seed}.joblib')
save_portable(model, model_path)
file_bytes = model_path.stat().st_size
assert file_bytes < 100_000_000, 'Model exceeds the 100 MB limit.'
prediction = model.predict(X_validation)
assert prediction.shape == (5000,) and np.isin(prediction, np.arange(10)).all()
# The new process sees only the model and input images, not training definitions.
with tempfile.TemporaryDirectory(dir=Path.cwd()) as directory:
    directory = Path(directory)
    np.save(directory / 'images.npy', X_validation[:37])
    check_code = ('import joblib,numpy as np; '
        + 'm=joblib.load(' + repr(str(model_path.resolve())) + '); '
        + "np.save('predictions.npy',m.predict(np.load('images.npy')))")
    subprocess.run([sys.executable, '-I', '-c', check_code], cwd=directory, check=True)
    np.testing.assert_array_equal(prediction[:37], np.load(directory / 'predictions.npy'))
row = {'seed': seed, 'validation_accuracy': float(accuracy_score(y_validation, prediction)),
       'development_accuracy': float(accuracy_score(y_validation[dev_indices], prediction[dev_indices])),
       'audit_accuracy': float(accuracy_score(y_validation[audit_indices], prediction[audit_indices])),
       'model_bytes': file_bytes, 'member_seconds': member_seconds,
       'export_evaluation_seconds': round(time.perf_counter()-started, 2),
       'sha256': hashlib.sha256(model_path.read_bytes()).hexdigest()}
results.append(row)
print(json.dumps(row, indent=2))
del model
_ = gc.collect()


{
  "seed": 42,
  "validation_accuracy": 0.6694,
  "development_accuracy": 0.668,
  "audit_accuracy": 0.6726666666666666,
  "model_bytes": 43200034,
  "member_seconds": [
    246.38,
    376.29,
    476.27,
    712.08,
    440.86,
    372.18,
    324.46
  ],
  "export_evaluation_seconds": 58.23,
  "sha256": "bc5cd45a86843b106d2a42ae9bce2d21dbbe5a6e3ef682b941d10cf71ebedf79"
}


## Export and check seed 2026

In [7]:
seed = 2026
started = time.perf_counter()
model, member_seconds = fitted_models.pop(0)
model_path = Path(f'model_seed{seed}.joblib')
save_portable(model, model_path)
file_bytes = model_path.stat().st_size
assert file_bytes < 100_000_000, 'Model exceeds the 100 MB limit.'
prediction = model.predict(X_validation)
assert prediction.shape == (5000,) and np.isin(prediction, np.arange(10)).all()
# The new process sees only the model and input images, not training definitions.
with tempfile.TemporaryDirectory(dir=Path.cwd()) as directory:
    directory = Path(directory)
    np.save(directory / 'images.npy', X_validation[:37])
    check_code = ('import joblib,numpy as np; '
        + 'm=joblib.load(' + repr(str(model_path.resolve())) + '); '
        + "np.save('predictions.npy',m.predict(np.load('images.npy')))")
    subprocess.run([sys.executable, '-I', '-c', check_code], cwd=directory, check=True)
    np.testing.assert_array_equal(prediction[:37], np.load(directory / 'predictions.npy'))
row = {'seed': seed, 'validation_accuracy': float(accuracy_score(y_validation, prediction)),
       'development_accuracy': float(accuracy_score(y_validation[dev_indices], prediction[dev_indices])),
       'audit_accuracy': float(accuracy_score(y_validation[audit_indices], prediction[audit_indices])),
       'model_bytes': file_bytes, 'member_seconds': member_seconds,
       'export_evaluation_seconds': round(time.perf_counter()-started, 2),
       'sha256': hashlib.sha256(model_path.read_bytes()).hexdigest()}
results.append(row)
print(json.dumps(row, indent=2))
del model
_ = gc.collect()


{
  "seed": 2026,
  "validation_accuracy": 0.6734,
  "development_accuracy": 0.6705714285714286,
  "audit_accuracy": 0.68,
  "model_bytes": 43224233,
  "member_seconds": [
    246.7,
    376.33,
    477.54,
    711.8,
    441.33,
    372.4,
    325.18
  ],
  "export_evaluation_seconds": 57.67,
  "sha256": "80ab213b2d200dc8f7bd2e646b20a388c582f78ee73d7f32ec485c6c61aa4e01"
}


## Export and check seed 3102

In [8]:
seed = 3102
started = time.perf_counter()
model, member_seconds = fitted_models.pop(0)
model_path = Path(f'model_seed{seed}.joblib')
save_portable(model, model_path)
file_bytes = model_path.stat().st_size
assert file_bytes < 100_000_000, 'Model exceeds the 100 MB limit.'
prediction = model.predict(X_validation)
assert prediction.shape == (5000,) and np.isin(prediction, np.arange(10)).all()
# The new process sees only the model and input images, not training definitions.
with tempfile.TemporaryDirectory(dir=Path.cwd()) as directory:
    directory = Path(directory)
    np.save(directory / 'images.npy', X_validation[:37])
    check_code = ('import joblib,numpy as np; '
        + 'm=joblib.load(' + repr(str(model_path.resolve())) + '); '
        + "np.save('predictions.npy',m.predict(np.load('images.npy')))")
    subprocess.run([sys.executable, '-I', '-c', check_code], cwd=directory, check=True)
    np.testing.assert_array_equal(prediction[:37], np.load(directory / 'predictions.npy'))
row = {'seed': seed, 'validation_accuracy': float(accuracy_score(y_validation, prediction)),
       'development_accuracy': float(accuracy_score(y_validation[dev_indices], prediction[dev_indices])),
       'audit_accuracy': float(accuracy_score(y_validation[audit_indices], prediction[audit_indices])),
       'model_bytes': file_bytes, 'member_seconds': member_seconds,
       'export_evaluation_seconds': round(time.perf_counter()-started, 2),
       'sha256': hashlib.sha256(model_path.read_bytes()).hexdigest()}
results.append(row)
print(json.dumps(row, indent=2))
del model
_ = gc.collect()


{
  "seed": 3102,
  "validation_accuracy": 0.6672,
  "development_accuracy": 0.6662857142857143,
  "audit_accuracy": 0.6693333333333333,
  "model_bytes": 43338890,
  "member_seconds": [
    247.01,
    376.31,
    477.23,
    710.89,
    441.83,
    373.67,
    324.86
  ],
  "export_evaluation_seconds": 59.52,
  "sha256": "28d1049d45f1b638dc448abb523b5f970330ccea5623d45aa5da09b1ac483533"
}


## Final validation summary
These are provided-validation results, not hidden-test results. The selection-development subset is optimistic; the reserved audit subset is the independent estimate. Final training uses all 50,000 training images.

In [9]:
summary = {'environment': versions, 'configuration': CONFIGURATION, 'models': results,
    'mean_validation_accuracy': float(np.mean([r['validation_accuracy'] for r in results])),
    'mean_audit_accuracy': float(np.mean([r['audit_accuracy'] for r in results])),
    'all_models_under_100MB': all(r['model_bytes'] < 100_000_000 for r in results),
    'fresh_process_loading_passed': True}
print(json.dumps(summary, indent=2))
Path('results_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')


{
  "environment": {
    "python": "3.14.6",
    "numpy": "2.4.6",
    "scipy": "1.18.0",
    "scikit-learn": "1.9.0",
    "joblib": "1.5.3"
  },
  "configuration": {
    "selection_metric": "3500-image official development accuracy",
    "selection_accuracy": 0.6677,
    "audit_used_for_selection": false,
    "training_threads": 1,
    "tiny_value_threshold": 1e-20,
    "horizontal_tta": true,
    "weights": [
      1,
      1,
      1,
      1,
      1,
      1,
      1
    ],
    "members": [
      {
        "name": "raw",
        "clusters": 512,
        "patch_size": 6,
        "n_patches": 60000,
        "smooth_sigma": 0,
        "augmentation_mode": "robust",
        "augmentation_ratio": 2.0,
        "hidden_layers": [
          384,
          128
        ],
        "alpha": 5.0,
        "learning_rate": 0.0005,
        "epochs": 5
      },
      {
        "name": "strong",
        "clusters": 512,
        "patch_size": 6,
        "n_patches": 60000,
        "smooth_sigma": 0,

4703